### Imports, loading data and methods from features.py

In [40]:
from features import *
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import wandb


data = pd.read_csv('train.csv')

cryosleep(data)
cabin(data)
passenger(data)
vip(data)

X = data.drop(columns=['PassengerId', 'Num', 'Name', 'Transported', 'Cabin', 'Group'])
y = data['Transported']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)
X_train, medians = money_columns(X_train)
X_test, _ = money_columns(X_test, medians)

### Categorical and Numerical Features

In [41]:
cat_col = X.select_dtypes(include=['object', 'str']).columns
num_col = X.select_dtypes(include=['number']).columns
print(num_col)

Index(['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall',
       'Spa', 'VRDeck', 'MoneySpend', 'GroupSize'],
      dtype='str')


### Categorical pipeline + OneHotEncoder, SimpleImputer, StandardScaler

In [42]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

age_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

### ColumnTransformer

In [43]:
preprocessor = ColumnTransformer([
    ('cat', cat_pipeline, cat_col),
    ('age', age_pipeline, ['Age']),
    ('vip', SimpleImputer(strategy='most_frequent'), ['VIP']),
    ('money', StandardScaler(), ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'MoneySpend']),
], remainder='passthrough')

X_train_preproc = preprocessor.fit_transform(X_train)
X_test_preproc = preprocessor.transform(X_test)

X_train_tensor = torch.tensor(X_train_preproc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test_preproc, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

y_train_tensor.shape

torch.Size([6085, 1])

### DataLoaders

In [44]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### MLP

In [45]:
input_dim = X_train_tensor.shape[1]

class Model(nn.Module):
    def __init__(self, input_dim, hidden_sizes, output_dim=1, activation=nn.ReLU):
        super().__init__()
        layers = []
        prev_size = input_dim

        for size in hidden_sizes:
            layers.append(nn.Linear(prev_size, size))
            layers.append(activation())
            prev_size = size

        layers.append(nn.Linear(prev_size, output_dim))
        self.neural_net = nn.Sequential(*layers)

    def forward(self, x):
        result = self.neural_net(x)
        return result

### Training Loop

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_model(lr, hidden_sizes, optimizer_name='SGD', activation=nn.ReLU):
    arch_labels = "-".join(map(str, hidden_sizes))
    wandb.init(
        project="Spaceship",
        name=f'{optimizer_name}-lr{lr}-{activation.__name__}-{arch_labels}',
        config={
            'lr': lr,
            'epochs': 20,
            'batch_size': 64,
            'optimizer': optimizer_name,
            'activation': activation.__name__,
            'hidden_sizes': hidden_sizes
        }
    )

    model = Model(input_dim=input_dim, hidden_sizes=hidden_sizes, activation=activation).to(device)
    criterion = nn.BCEWithLogitsLoss()

    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr)
    else:
        raise ValueError()

    for epoch in range(20):
        model.train()
        running_loss = 0.0
        for X, y in train_dataloader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            result = model(X)
            loss = criterion(result, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()

        avg_loss = running_loss / len(train_dataloader)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in test_dataloader:
                X, y = X.to(device), y.to(device)
                result = model(X)
                prediction = (result > 0).int()
                correct += (prediction == y).sum().item()
                total += len(y)
        accuracy = correct / total

        wandb.log({'loss': avg_loss, 'Accuracy': accuracy, 'epoch': epoch})
    wandb.finish()

    return model

train_model(0.01, [64, 64, 64], optimizer_name='AdamW', activation=nn.ReLU)

wandb: setting up run 9ua6oxnt
wandb: Tracking run with wandb version 0.28.2
wandb: Run data is saved locally in C:\Users\kubak\Documents\Programming\PycharmProjects\MachineLearning\Spaceship Titanic\wandb\run-20260901_164444-9ua6oxnt
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run AdamW-lr0.01-ReLU-64-64-64
wandb:  View project at https://wandb.ai/jacob_21-agh-univeristy-of-krakow/Spaceship
wandb:  View run at https://wandb.ai/jacob_21-agh-univeristy-of-krakow/Spaceship/runs/9ua6oxnt
wandb: updating run metadata
wandb: uploading config.yaml
wandb: uploading history steps 0-19, summary
wandb: 
wandb: Run history:
wandb: Accuracy ▇▃▂▆▅▁▆▆▇▄▆▇▇▄▆▃▄▇▇█
wandb:    epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:     loss █▄▄▄▃▃▃▂▂▃▃▂▂▂▂▂▂▁▁▁
wandb: 
wandb: Run summary:
wandb: Accuracy 0.79563
wandb:    epoch 19
wandb:     loss 0.36657
wandb: 
wandb:  View run AdamW-lr0.01-ReLU-64-64-64 at: https://wandb.ai/jacob_21-agh-univeristy-of-krakow/Spaceship/runs/9ua6oxnt
wandb:  View project at

### Kaggle Submission

In [46]:
X_full, medians = money_columns(X)

X_full_preproc = preprocessor.fit_transform(X_full)

X_full_tensor = torch.tensor(X_full_preproc, dtype=torch.float32).to(device)
y_full_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)

full_dataset = TensorDataset(X_full_tensor, y_full_tensor)
full_dataloader = DataLoader(full_dataset, batch_size=64, shuffle=True)

input_dim_final = X_full_preproc.shape[1]

In [47]:
def train_final_model(lr, hidden_sizes, dataloader, input_dim, optimizer_name="AdamW", activation=nn.ReLU, epochs=20):
    model = Model(input_dim, hidden_sizes, activation=activation).to(device)
    criterion = nn.BCEWithLogitsLoss()

    if optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr)
    else:
        raise ValueError()

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            result = model(X_batch)
            loss = criterion(result, y_batch)
            loss.backward()
            optimizer.step()
    return model

final_model = train_final_model(
    lr=0.01, hidden_sizes=[64, 64, 64], dataloader=full_dataloader,
    input_dim=input_dim_final, optimizer_name='AdamW', activation=nn.ReLU, epochs=20)

test_data = pd.read_csv("test.csv")
cryosleep(test_data)
cabin(test_data)
passenger(test_data)
vip(test_data)
test_data, _ = money_columns(test_data, medians)

passenger_id = test_data['PassengerId']
X_test_final = test_data.drop(columns=['PassengerId', 'Num', 'Name', 'Cabin', 'Group'])

X_test_preproc = preprocessor.transform(X_test_final)
X_test_tensor = torch.tensor(X_test_preproc, dtype=torch.float32).to(device)

final_model.eval()
with torch.no_grad():
    result = final_model(X_test_tensor)
    predictions = (result > 0).int().cpu().numpy().flatten()

submission = pd.DataFrame({
    'PassengerId': passenger_id,
    'Transported': predictions.astype(bool)
})
submission.to_csv('submission.csv', index=False)